# 🐍 Python Dataclasses — AI Engineer ke liye Practice Notebook

**Language:** Roman Urdu notes + chalne wala code
**Requirement:** Python 3.10+ (sirf aakhri section ke liye `pip install pydantic`)

### Is notebook mein kya hai
1. Dataclass kya hai aur kyun
2. Basics: `@dataclass`
3. Default values aur `field(default_factory=...)`
4. `__post_init__`: chhoti validation aur derived values
5. `frozen=True`: config lock karna
6. `asdict()`, `astuple()`, `replace()`
7. `kw_only=True` aur `field()` ke options
8. Inheritance
9. **Real AI use cases** (LLM config, chat history, RAG chunks, tool calls, training config, agent state, eval results)
10. Dataclass + Pydantic ek saath
11. Common galtiyan
12. **Practice exercises** (solutions ke saath)
13. Summary: kahan use karein, kahan nahi

> **Tip:** Har cell ko `Shift + Enter` se chalayein aur output dekhein. Phir code change kar ke experiment karein.

## 1. Dataclass kya hai aur kyun?

Dataclass Python ka **built-in** feature hai (`from dataclasses import dataclass`). Kuch install nahi karna.

Jab aap ki class ka main kaam **data rakhna** ho (logic kam, fields zyada), to dataclass aap ka boilerplate code khud likh deta hai:

| Ye method | Kya karta hai | Dataclass khud banata hai? |
|---|---|---|
| `__init__` | Object banana | ✅ |
| `__repr__` | Print karne par saaf output | ✅ |
| `__eq__` | `obj1 == obj2` compare | ✅ |

**Yaad rakhein:** Dataclass **validation nahi karta**. Type hints sirf documentation hain. Validation chahiye to Pydantic.

### Pehle normal class dekhein (bina dataclass)

In [1]:
class WatchNormal:
    def __init__(self, name, price, brand):
        self.name = name
        self.price = price
        self.brand = brand

    def __repr__(self):
        return f"WatchNormal(name={self.name!r}, price={self.price}, brand={self.brand!r})"

    def __eq__(self, other):
        return (self.name, self.price, self.brand) == (other.name, other.price, other.brand)

print(WatchNormal("Classic", 4500, "Casio"))

WatchNormal(name='Classic', price=4500, brand='Casio')


### Ab wahi cheez dataclass ke saath

In [2]:
from dataclasses import dataclass

@dataclass
class Watch:
    name: str
    price: int
    brand: str

w1 = Watch("Classic", 4500, "Casio")
w2 = Watch("Classic", 4500, "Casio")

print(w1)          # __repr__ khud bana
print(w1 == w2)    # __eq__ khud bana -> True
print(w1.price)    # normal attribute access

Watch(name='Classic', price=4500, brand='Casio')
True
4500


### ⚠️ Validation nahi hoti — ye zaroor dekhein
Neeche price mein string aur brand mein number de rahe hain. Koi error **nahi** aaye ga.

In [3]:
galat = Watch("Classic", "bohat mehngi", 123)
print(galat)
print(type(galat.price))   # <class 'str'>  -> Python ne check nahi kiya

Watch(name='Classic', price='bohat mehngi', brand=123)
<class 'str'>


## 2. Default values

Fields ko default value de sakte hain. **Rule:** jin fields ki default value nahi hai wo **pehle** aayein, default wali **baad** mein.

In [4]:
@dataclass
class LLMSettings:
    model: str                     # default nahi -> pehle
    temperature: float = 0.7       # default wali -> baad mein
    max_tokens: int = 500

s = LLMSettings("gpt-4o-mini")
print(s)

s2 = LLMSettings("claude-sonnet", temperature=0.2)
print(s2)

LLMSettings(model='gpt-4o-mini', temperature=0.7, max_tokens=500)
LLMSettings(model='claude-sonnet', temperature=0.2, max_tokens=500)


## 3. `field(default_factory=...)` — sab se zaroori cheez

List, dict ya set ko **direct default** nahi de sakte (`messages: list = []` error dega).
Wajah: agar Python allow karta to **saare objects ek hi list share karte** — bohat bura bug.

Hal: `field(default_factory=list)` — har naye object ke liye **nayi** list banti hai.

In [5]:
from dataclasses import field

try:
    @dataclass
    class Galat:
        messages: list = []
except ValueError as e:
    print("Error aaya:", e)

Error aaya: mutable default <class 'list'> for field messages is not allowed: use default_factory


In [6]:
@dataclass
class ChatSession:
    user_id: str
    messages: list = field(default_factory=list)
    metadata: dict = field(default_factory=dict)

a = ChatSession("user_1")
b = ChatSession("user_2")

a.messages.append("Salam!")

print("a:", a.messages)
print("b:", b.messages)   # b ki list alag hai, khali hai -> sahi

a: ['Salam!']
b: []


`default_factory` mein koi bhi function de sakte hain, jaise unique ID ya current time:

In [7]:
import uuid
from datetime import datetime

@dataclass
class Request:
    prompt: str
    request_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    created_at: datetime = field(default_factory=datetime.now)

print(Request("Hello"))
print(Request("Hello"))   # har baar alag id aur time

Request(prompt='Hello', request_id='0ced9f4f', created_at=datetime.datetime(2026, 10, 30, 11, 9, 59, 333255))
Request(prompt='Hello', request_id='b3c350a6', created_at=datetime.datetime(2026, 10, 30, 11, 9, 59, 334274))


## 4. `__post_init__` — object banne ke baad kuch karna

`__init__` dataclass khud banata hai, to apna code kahan likhein? **`__post_init__`** mein.
Ye object banne ke **foran baad** chalta hai.

Use:
- Chhoti validation (range check)
- Derived/calculated fields banana
- Values clean karna (strip, lower)

In [8]:
@dataclass
class TrainConfig:
    learning_rate: float
    epochs: int
    batch_size: int = 32

    def __post_init__(self):
        if self.learning_rate <= 0:
            raise ValueError("learning_rate positive honi chahiye")
        if self.epochs < 1:
            raise ValueError("epochs kam az kam 1 hon")

print(TrainConfig(0.001, 10))

try:
    TrainConfig(-0.5, 10)
except ValueError as e:
    print("Error:", e)

TrainConfig(learning_rate=0.001, epochs=10, batch_size=32)
Error: learning_rate positive honi chahiye


### Derived field: `field(init=False)`
Kuch fields user nahi deta, hum khud calculate karte hain. Unhein `init=False` rakhein.

In [9]:
@dataclass
class DocumentChunk:
    text: str
    source: str
    word_count: int = field(init=False)   # user nahi de ga

    def __post_init__(self):
        self.text = self.text.strip()
        self.word_count = len(self.text.split())

chunk = DocumentChunk("   Suparnova par COD available hai.   ", "faq.txt")
print(chunk)

DocumentChunk(text='Suparnova par COD available hai.', source='faq.txt', word_count=5)


## 5. `frozen=True` — object ko lock karna (immutable)

Config ek baar ban jaye to beech mein koi galti se change na kare. `frozen=True` se assignment par error aata hai.

**Bonus:** frozen dataclass **hashable** hota hai, yani dict ki key ya set mein use ho sakta hai (cache ke liye kaam ki cheez).

In [10]:
from dataclasses import FrozenInstanceError

@dataclass(frozen=True)
class ModelConfig:
    model_name: str = "gpt-4o-mini"
    temperature: float = 0.7

cfg = ModelConfig()

try:
    cfg.temperature = 1.5
except FrozenInstanceError as e:
    print("Change nahi kar sakte:", e)

# Hashable -> dict key ban sakta hai (jaise response cache)
cache = {cfg: "pichla jawab"}
print(cache[ModelConfig()])   # same values -> same key

Change nahi kar sakte: cannot assign to field 'temperature'
pichla jawab


## 6. `asdict()`, `astuple()`, `replace()`

| Function | Kaam | AI mein kahan |
|---|---|---|
| `asdict(obj)` | Dict bana deta hai (nested bhi) | JSON save, API ko bhejna, wandb/MLflow logging |
| `astuple(obj)` | Tuple bana deta hai | CSV row, DB insert |
| `replace(obj, x=..)` | Copy bana kar kuch values badal deta hai | Frozen config ka naya version, hyperparameter sweep |

In [11]:
from dataclasses import asdict, astuple, replace
import json

cfg = ModelConfig(model_name="claude-haiku", temperature=0.3)

d = asdict(cfg)
print(d)
print(json.dumps(d))
print(astuple(cfg))

# Frozen object change nahi hota, lekin replace se naya bana sakte hain
creative = replace(cfg, temperature=1.0)
print("purana:", cfg)
print("naya  :", creative)

{'model_name': 'claude-haiku', 'temperature': 0.3}
{"model_name": "claude-haiku", "temperature": 0.3}
('claude-haiku', 0.3)
purana: ModelConfig(model_name='claude-haiku', temperature=0.3)
naya  : ModelConfig(model_name='claude-haiku', temperature=1.0)


### Hyperparameter sweep: `replace()` ka asli fayda

In [12]:
base = TrainConfig(learning_rate=0.001, epochs=5)

experiments = [replace(base, learning_rate=lr) for lr in [0.1, 0.01, 0.001, 0.0001]]
for e in experiments:
    print(asdict(e))

{'learning_rate': 0.1, 'epochs': 5, 'batch_size': 32}
{'learning_rate': 0.01, 'epochs': 5, 'batch_size': 32}
{'learning_rate': 0.001, 'epochs': 5, 'batch_size': 32}
{'learning_rate': 0.0001, 'epochs': 5, 'batch_size': 32}


### JSON se wapas dataclass banana

In [13]:
json_text = '{"model_name": "gpt-4o", "temperature": 0.2}'
loaded = ModelConfig(**json.loads(json_text))   # ** se dict ko arguments bana diya
print(loaded)

ModelConfig(model_name='gpt-4o', temperature=0.2)


## 7. `kw_only=True` aur `field()` ke kuch options

**`kw_only=True`** (Python 3.10+): Har field ko naam se dena parta hai. Fayde:
- Default order wala rule khatam
- Code parhne mein saaf: `Config(lr=0.01)` vs `Config(0.01)`

**`field()` options jo kaam aate hain:**
- `repr=False` → print mein na dikhe (API keys ke liye 🔐)
- `compare=False` → `==` mein shamil na ho (timestamps ke liye)

In [14]:
@dataclass(kw_only=True)
class APIClientConfig:
    base_url: str = "https://api.example.com"
    api_key: str = field(repr=False)                       # print mein chhupa
    timeout: int = 30
    created_at: datetime = field(default_factory=datetime.now, compare=False)

c = APIClientConfig(api_key="sk-secret-123")
print(c)            # api_key nazar nahi aa rahi

# try:
#     APIClientConfig("https://x.com", "sk-1")   # positional allowed nahi
# except TypeError as e:
#     print("Error:", e)

APIClientConfig(base_url='https://api.example.com', timeout=30, created_at=datetime.datetime(2026, 10, 30, 11, 18, 24, 593344))


## 8. Inheritance — base config ko extend karna

Common settings ek base class mein, specific settings child class mein.

In [16]:
@dataclass(kw_only=True)
class BaseConfig:
    project_name: str = "whatsapp-bot"
    seed: int = 42
    debug: bool = False

@dataclass(kw_only=True)
class LLMConfig(BaseConfig):
    model: str = "gpt-4o-mini"
    temperature: float = 0.5

@dataclass(kw_only=True)
class EmbeddingConfig(BaseConfig):
    model: str = "text-embedding-3-small"
    dimensions: int = 1536
    chunk_size: int = 500

print(LLMConfig(debug=True))
print(EmbeddingConfig(chunk_size=800))

LLMConfig(project_name='whatsapp-bot', seed=42, debug=True, model='gpt-4o-mini', temperature=0.5)
EmbeddingConfig(project_name='whatsapp-bot', seed=42, debug=False, model='text-embedding-3-small', dimensions=1536, chunk_size=800)


## 10. Dataclass + Pydantic ek saath (asli project pattern)

**Golden rule:**
- **Bahar se aane wala data** (API request, webhook, user input, LLM ka JSON output) → **Pydantic** (validation zaroori)
- **Andar ka data** (config, internal state, function se function) → **Dataclass** (halka aur fast)

Neeche ek WhatsApp webhook ka misal: bahar ka JSON Pydantic validate karta hai, phir andar dataclass mein convert ho jata hai.

> Pehle `pip install pydantic` kar lein agar install nahi hai.

In [20]:
from pydantic import BaseModel, Field, ValidationError

# ---- Bahar ka data: Pydantic ----
class IncomingWebhook(BaseModel):
    phone: str = Field(min_length=10)
    text: str = Field(min_length=1)
    timestamp: int

# ---- Andar ka data: Dataclass ----
@dataclass
class InternalMessage:
    phone: str
    text: str
    received_at: datetime

def handle_webhook(raw: dict) -> InternalMessage | None:
    try:
        data = IncomingWebhook(**raw)          # validation yahan
    except ValidationError as e:
        print("❌ Galat data:", e.errors()[0]["msg"])
        return None
    return InternalMessage(data.phone, data.text.strip(),
                           datetime.fromtimestamp(data.timestamp))

print(handle_webhook({"phone": "923001234567", "text": " Salam ", "timestamp": "1761125000"}))
print(handle_webhook({"phone": "123", "text": "hi", "timestamp": 1761125000}))

InternalMessage(phone='923001234567', text='Salam', received_at=datetime.datetime(2025, 10, 22, 14, 23, 20))
❌ Galat data: String should have at least 10 characters
None


Note: upar `timestamp` string `"1761125000"` thi, Pydantic ne khud `int` bana di. Dataclass ye nahi karta.

### Bonus: Pydantic ka bhi ek `dataclass` hai
Agar dataclass ki syntax pasand hai lekin validation bhi chahiye:

In [ ]:
from pydantic.dataclasses import dataclass as pyd_dataclass

@pyd_dataclass
class ValidatedWatch:
    name: str
    price: int

print(ValidatedWatch("Classic", "4500"))   # "4500" -> 4500 convert
try:
    ValidatedWatch("Classic", "bohat mehngi")
except ValidationError as e:
    print("Pydantic dataclass ne pakar liya:", e.errors()[0]["msg"])

## 11. Common galtiyan (aur unka hal)

| Galti | Kya hota hai | Hal |
|---|---|---|
| `items: list = []` | `ValueError` | `field(default_factory=list)` |
| Default wala field pehle, bina default wala baad mein | `TypeError: non-default argument follows default argument` | Order theek karein ya `kw_only=True` |
| Sochna ke type hints check hote hain | Galat data chupke se andar aa jata hai | `__post_init__` ya Pydantic |
| Frozen object mein value badalna | `FrozenInstanceError` | `replace(obj, field=new)` |
| API key print/log ho jana | Security risk | `field(repr=False)` |
| `asdict()` mein `datetime` → `json.dumps` fail | `TypeError: not JSON serializable` | `json.dumps(asdict(x), default=str)` |

In [ ]:
# Aakhri galti ka demo aur hal
req = Request("Hello")
try:
    json.dumps(asdict(req))
except TypeError as e:
    print("Error:", e)

print(json.dumps(asdict(req), default=str))   # hal

## ⚡ Bonus: `slots=True` (memory bachana)
Jab lakhon objects banane hon (jaise millions of chunks), `slots=True` memory kam karta hai aur attribute access thora tez hota hai. Galti se naya attribute bhi nahi lag sakta.

In [ ]:
@dataclass(slots=True)
class LightChunk:
    text: str
    score: float

c = LightChunk("hello", 0.9)
try:
    c.galat_attribute = 1
except AttributeError as e:
    print("slots ne rok diya:", e)

## 13. 📌 Summary: Kahan use karein, kahan nahi

### ✅ Dataclass use karein jab
| Situation | Misal |
|---|---|
| App / model ki settings | `LLMConfig`, `FineTuneConfig`, `BotConfig` |
| Chat history aur messages | `Message`, `Conversation` |
| RAG pipeline ka data | `Chunk`, `SearchResult` |
| Agent ka internal state | `AgentState`, `ToolResult` |
| Logging / metrics | `TokenUsage`, `LLMResponse`, `EvalReport` |
| Function se function data pass karna | Tuple ya dict ki jagah naam wale fields |

### ✅ Pydantic use karein jab
| Situation | Misal |
|---|---|
| API request / response (FastAPI) | Webhook body, REST endpoints |
| User input | Forms, chatbot mein user ka data |
| LLM structured output | JSON mode, `response_format`, LangChain `with_structured_output` |
| `.env` settings | `pydantic-settings` ka `BaseSettings` |
| External JSON / files | Shopify webhook, third-party APIs |

### ❌ Dataclass ki zaroorat nahi jab
- Class mein zyada tar **logic / behavior** hai, data kam (normal class likhein)
- Sirf 2-3 values ek baar return karni hain (`NamedTuple` ya tuple kaafi hai)

### Cheat sheet
```python
from dataclasses import dataclass, field, asdict, replace

@dataclass(frozen=True, kw_only=True, slots=True)
class X:
    a: str
    b: list = field(default_factory=list)
    secret: str = field(repr=False, default="")
    derived: int = field(init=False, default=0)

    def __post_init__(self): ...
```

**Agla qadam:** Apne WhatsApp bot ki settings ko ek `frozen` dataclass mein le jayein, aur conversation history ke liye Section 9.1 wala `Conversation` pattern use karein. 🚀